# NSE Trading Agent — CatBoost Model Training (Nifty 250, Daily Data)

**What this does:**
- Fetches 2 years of daily candle data for 250 NSE stocks (~125K+ samples)
- Engineers 45+ technical features (multi-timeframe, momentum, volume, volatility)
- Trains a CatBoost classifier with walk-forward validation
- Target: Will price move up by ≥1.5% within the next 3 trading days?
- Exports model compatible with `predictor.py`

> **Note:** yfinance limits 15-min data to 60 days max. Daily data over 2 years gives far more training samples and more reliable patterns.

### Instructions:
1. **Runtime → Change runtime type → T4 GPU** (CatBoost supports GPU training)
2. Click **Run All**
3. Download `predictor_catboost.pkl` and `feature_cols.json` → place in `ai-trading-agent/models/`

In [ ]:
!pip install -q yfinance pandas numpy ta catboost scikit-learn

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import pickle
import json
import os
import time
import catboost as cb
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import TimeSeriesSplit
import warnings
warnings.filterwarnings('ignore')

print(f"CatBoost version: {cb.__version__}")
print(f"GPU available: {cb.CatBoostClassifier(task_type='GPU', iterations=1).is_gpu_available()}" if hasattr(cb.CatBoostClassifier, 'is_gpu_available') else 'Check manually')

## 1. Nifty 250 Stock List

In [ ]:
# Nifty 250 stocks (Nifty 50 + Nifty Next 50 + Nifty Midcap 150 top picks)
NIFTY_250 = [
    # ── Nifty 50 ──
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "ICICIBANK.NS", "INFY.NS",
    "SBIN.NS", "BHARTIARTL.NS", "HINDUNILVR.NS", "ITC.NS", "LT.NS",
    "BAJFINANCE.NS", "HCLTECH.NS", "AXISBANK.NS", "KOTAKBANK.NS", "ASIANPAINT.NS",
    "MARUTI.NS", "SUNPHARMA.NS", "TITAN.NS", "ULTRACEMCO.NS", "BAJAJFINSV.NS",
    "WIPRO.NS", "NESTLEIND.NS", "ONGC.NS", "POWERGRID.NS", "M&M.NS",
    "NTPC.NS", "TATASTEEL.NS", "COALINDIA.NS", "HINDALCO.NS", "TATAMOTORS.NS",
    "JSWSTEEL.NS", "ADANIPORTS.NS", "GRASIM.NS", "ADANIENT.NS", "TECHM.NS",
    "BRITANNIA.NS", "INDUSINDBK.NS", "BAJAJ-AUTO.NS", "EICHERMOT.NS", "CIPLA.NS",
    "DIVISLAB.NS", "DRREDDY.NS", "HEROMOTOCO.NS", "APOLLOHOSP.NS", "HDFCLIFE.NS",
    "SBILIFE.NS", "BPCL.NS", "SHREECEM.NS", "LTIM.NS", "TRENT.NS",
    # ── Nifty Next 50 ──
    "ADANIGREEN.NS", "ADANIPOWER.NS", "AMBUJACEM.NS", "AUROPHARMA.NS", "BANKBARODA.NS",
    "BEL.NS", "BERGEPAINT.NS", "BIOCON.NS", "BOSCHLTD.NS", "CANBK.NS",
    "CHOLAFIN.NS", "COLPAL.NS", "CONCOR.NS", "CUMMINSIND.NS", "DABUR.NS",
    "DLF.NS", "GAIL.NS", "GODREJCP.NS", "GODREJPROP.NS", "HAL.NS",
    "HAVELLS.NS", "ICICIPRULI.NS", "IDEA.NS", "IDFCFIRSTB.NS", "INDHOTEL.NS",
    "INDUSTOWER.NS", "IOC.NS", "IRCTC.NS", "IRFC.NS", "JIOFIN.NS",
    "JSWENERGY.NS", "JUBLFOOD.NS", "LICI.NS", "LUPIN.NS", "MARICO.NS",
    "MCDOWELL-N.NS", "MPHASIS.NS", "NAUKRI.NS", "NHPC.NS", "NMDC.NS",
    "OBEROIRLTY.NS", "OFSS.NS", "PAYTM.NS", "PGHH.NS", "PIDILITIND.NS",
    "PNB.NS", "POLYCAB.NS", "RECLTD.NS", "SBICARD.NS", "SIEMENS.NS",
    # ── Nifty Midcap Select ──
    "AARTI.NS", "ABB.NS", "ABCAPITAL.NS", "ACC.NS", "ALKEM.NS",
    "ASHOKLEY.NS", "ASTRAL.NS", "ATUL.NS", "AUBANK.NS", "BALKRISIND.NS",
    "BATAINDIA.NS", "BHEL.NS", "CANFINHOME.NS", "CHAMBLFERT.NS", "COFORGE.NS",
    "COROMANDEL.NS", "CROMPTON.NS", "CUB.NS", "CUMMINSIND.NS", "CYIENT.NS",
    "DEEPAKNTR.NS", "DELTACORP.NS", "DEVYANI.NS", "DIXON.NS", "ESCORTS.NS",
    "EXIDEIND.NS", "FEDERALBNK.NS", "FORTIS.NS", "GLENMARK.NS", "GMRINFRA.NS",
    "GNFC.NS", "GSPL.NS", "HFCL.NS", "HINDPETRO.NS", "HUDCO.NS",
    "IBREALEST.NS", "IDFC.NS", "IEX.NS", "INDIANB.NS", "INDIAMART.NS",
    "IREDA.NS", "JKCEMENT.NS", "JSL.NS", "KAJARIACER.NS", "KEI.NS",
    "L&TFH.NS", "LALPATHLAB.NS", "LAURUSLABS.NS", "LICHSGFIN.NS", "LTTS.NS",
    # ── Midcap continued + Smallcap picks ──
    "MANAPPURAM.NS", "MFSL.NS", "MOTHERSON.NS", "MUTHOOTFIN.NS", "NAM-INDIA.NS",
    "NATIONALUM.NS", "NAUKRI.NS", "NAVINFLUOR.NS", "NBCC.NS", "NCC.NS",
    "NOCIL.NS", "PERSISTENT.NS", "PETRONET.NS", "PFC.NS", "PHOENIXLTD.NS",
    "PIIND.NS", "PRESTIGE.NS", "PVRINOX.NS", "RAMCOCEM.NS", "RBLBANK.NS",
    "RVNL.NS", "SAIL.NS", "SANOFI.NS", "SJVN.NS", "SOLARINDS.NS",
    "SRF.NS", "STAR.NS", "SUNDARMFIN.NS", "SUNDRMFAST.NS", "SUNTV.NS",
    "SUPREMEIND.NS", "SUZLON.NS", "SYNGENE.NS", "TATACHEM.NS", "TATACOMM.NS",
    "TATACONSUM.NS", "TATAELXSI.NS", "TATAPOWER.NS", "TORNTPHARM.NS", "TORNTPOWER.NS",
    "TTML.NS", "TVSMOTOR.NS", "UBL.NS", "UNIONBANK.NS", "UPL.NS",
    "VEDL.NS", "VOLTAS.NS", "WHIRLPOOL.NS", "YESBANK.NS", "ZEEL.NS",
    # ── Popular trading stocks ──
    "ZOMATO.NS", "DELHIVERY.NS", "NYKAA.NS", "POLICYBZR.NS", "CARTRADE.NS",
    "CLEAN.NS", "HAPPSTMNDS.NS", "ROUTE.NS", "KPITTECH.NS", "SONACOMS.NS",
    "TIINDIA.NS", "APLAPOLLO.NS", "IIFL.NS", "GUJGASLTD.NS", "ABFRL.NS",
    "PAGEIND.NS", "MUTHOOTFIN.NS", "MCX.NS", "IPCALAB.NS", "SUMICHEM.NS",
]

# Remove duplicates
NIFTY_250 = list(dict.fromkeys(NIFTY_250))
print(f"Total unique symbols: {len(NIFTY_250)}")

## 2. Fetch 2 Years of Daily Data

Daily data has no time limit on yfinance — reliable for all stocks.

In [ ]:
all_stock_data = {}
failed = []

for i, symbol in enumerate(NIFTY_250):
    if (i + 1) % 25 == 0 or i == 0:
        print(f"Fetching [{i+1}/{len(NIFTY_250)}]: {symbol}")
    try:
        df = yf.download(symbol, period="5y", interval="1d", progress=False)
        if not df.empty and len(df) > 200:
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.droplevel(1)
            all_stock_data[symbol] = df
        else:
            failed.append(symbol)
    except Exception as e:
        failed.append(symbol)

    # Rate limit
    if (i + 1) % 10 == 0:
        time.sleep(0.3)

print(f"\nSuccessfully fetched: {len(all_stock_data)} stocks")
print(f"Failed: {len(failed)} stocks")
if failed:
    print(f"Failed symbols: {failed[:20]}{'...' if len(failed) > 20 else ''}")
total_rows = sum(len(df) for df in all_stock_data.values())
print(f"Total daily candles: {total_rows:,}")

## 3. Feature Engineering (45+ features)

Multi-period indicators, momentum, volume profile, and volatility features on daily data.

In [ ]:
def engineer_features(df):
    """Create 45+ features from daily OHLCV data."""
    df = df.copy()
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']

    # ── RSI (multiple periods) ──
    df['rsi_14'] = ta.momentum.RSIIndicator(close, window=14).rsi()
    df['rsi_7'] = ta.momentum.RSIIndicator(close, window=7).rsi()
    df['rsi_21'] = ta.momentum.RSIIndicator(close, window=21).rsi()
    df['rsi_change'] = df['rsi_14'].diff()
    df['rsi_change_3'] = df['rsi_14'].diff(3)

    # ── EMAs ──
    df['ema_9'] = ta.trend.EMAIndicator(close, window=9).ema_indicator()
    df['ema_21'] = ta.trend.EMAIndicator(close, window=21).ema_indicator()
    df['ema_50'] = ta.trend.EMAIndicator(close, window=50).ema_indicator()
    df['ema_100'] = ta.trend.EMAIndicator(close, window=100).ema_indicator()
    df['ema_200'] = ta.trend.EMAIndicator(close, window=200).ema_indicator()
    df['ema_diff_9_21'] = (df['ema_9'] - df['ema_21']) / close
    df['ema_diff_21_50'] = (df['ema_21'] - df['ema_50']) / close
    df['ema_diff_50_200'] = (df['ema_50'] - df['ema_200']) / close
    df['price_vs_ema9'] = (close - df['ema_9']) / close
    df['price_vs_ema21'] = (close - df['ema_21']) / close
    df['price_vs_ema50'] = (close - df['ema_50']) / close
    df['price_vs_ema200'] = (close - df['ema_200']) / close

    # ── MACD ──
    macd = ta.trend.MACD(close)
    df['macd'] = macd.macd() / close
    df['macd_signal'] = macd.macd_signal() / close
    df['macd_diff'] = macd.macd_diff() / close
    df['macd_diff_change'] = df['macd_diff'].diff()

    # ── Bollinger Bands ──
    bb = ta.volatility.BollingerBands(close)
    df['bb_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / bb.bollinger_mavg()
    df['bb_position'] = (close - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband() + 1e-8)

    # ── ATR ──
    df['atr'] = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range() / close
    df['atr_7'] = ta.volatility.AverageTrueRange(high, low, close, window=7).average_true_range() / close

    # ── Stochastic ──
    stoch = ta.momentum.StochasticOscillator(high, low, close)
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()
    df['stoch_diff'] = df['stoch_k'] - df['stoch_d']

    # ── ADX (trend strength) ──
    adx = ta.trend.ADXIndicator(high, low, close, window=14)
    df['adx'] = adx.adx()
    df['adx_diff'] = adx.adx_pos() - adx.adx_neg()

    # ── Returns (multi-period) ──
    df['return_1'] = close.pct_change(1)
    df['return_3'] = close.pct_change(3)
    df['return_5'] = close.pct_change(5)
    df['return_10'] = close.pct_change(10)
    df['return_20'] = close.pct_change(20)

    # ── Volatility ──
    df['volatility_5'] = df['return_1'].rolling(5).std()
    df['volatility_10'] = df['return_1'].rolling(10).std()
    df['volatility_20'] = df['return_1'].rolling(20).std()
    df['vol_ratio_5_20'] = df['volatility_5'] / (df['volatility_20'] + 1e-8)

    # ── Volume features ──
    vol_sma_20 = volume.rolling(20).mean()
    df['volume_ratio'] = volume / (vol_sma_20 + 1)
    df['volume_change'] = volume.pct_change()
    df['volume_trend'] = volume.rolling(5).mean() / (volume.rolling(20).mean() + 1)

    # ── Price action ──
    df['high_low_range'] = (high - low) / close
    df['close_position'] = (close - low) / (high - low + 1e-8)
    df['upper_wick'] = (high - np.maximum(close, df['Open'])) / (high - low + 1e-8)
    df['lower_wick'] = (np.minimum(close, df['Open']) - low) / (high - low + 1e-8)
    df['body_size'] = abs(close - df['Open']) / (high - low + 1e-8)

    # ── Support / Resistance ──
    df['dist_to_high_20'] = (high.rolling(20).max() - close) / close
    df['dist_to_low_20'] = (close - low.rolling(20).min()) / close
    df['dist_to_high_50'] = (high.rolling(50).max() - close) / close
    df['dist_to_low_50'] = (close - low.rolling(50).min()) / close

    # ── RSI divergence (price vs RSI direction over last 5 days) ──
    df['rsi_divergence'] = np.sign(close.diff(5)) * np.sign(-df['rsi_14'].diff(5))

    # ── Day of week (cyclical) ──
    if hasattr(df.index, 'dayofweek'):
        df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 5)
        df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 5)
    else:
        df['dow_sin'] = 0
        df['dow_cos'] = 0

    return df


FEATURE_COLS = [
    # RSI
    'rsi_14', 'rsi_7', 'rsi_21', 'rsi_change', 'rsi_change_3',
    # EMA
    'ema_diff_9_21', 'ema_diff_21_50', 'ema_diff_50_200',
    'price_vs_ema9', 'price_vs_ema21', 'price_vs_ema50', 'price_vs_ema200',
    # MACD
    'macd', 'macd_signal', 'macd_diff', 'macd_diff_change',
    # Bollinger
    'bb_width', 'bb_position',
    # ATR
    'atr', 'atr_7',
    # Stochastic
    'stoch_k', 'stoch_d', 'stoch_diff',
    # ADX
    'adx', 'adx_diff',
    # Returns
    'return_1', 'return_3', 'return_5', 'return_10', 'return_20',
    # Volatility
    'volatility_5', 'volatility_10', 'volatility_20', 'vol_ratio_5_20',
    # Volume
    'volume_ratio', 'volume_change', 'volume_trend',
    # Price action
    'high_low_range', 'close_position', 'upper_wick', 'lower_wick', 'body_size',
    # Support/Resistance
    'dist_to_high_20', 'dist_to_low_20', 'dist_to_high_50', 'dist_to_low_50',
    # Divergence
    'rsi_divergence',
    # Time
    'dow_sin', 'dow_cos',
]

print(f"Feature count: {len(FEATURE_COLS)}")

## 4. Build Training Dataset

In [ ]:
TARGET_HORIZON = 5       # Look ahead 5 trading days (1 week)
TARGET_THRESHOLD = 0.02  # 2% move = clear directional signal

all_processed = []
symbol_counts = {}

for i, (symbol, df) in enumerate(all_stock_data.items()):
    if (i + 1) % 50 == 0:
        print(f"Processing [{i+1}/{len(all_stock_data)}]: {symbol}")

    try:
        featured = engineer_features(df)

        # Target: max price in next N days is >= threshold above current close
        future_max = featured['High'].rolling(TARGET_HORIZON).max().shift(-TARGET_HORIZON)
        featured['target'] = (future_max >= featured['Close'] * (1 + TARGET_THRESHOLD)).astype(int)

        featured['symbol'] = symbol

        # Drop rows with NaN in features or target
        valid = featured.dropna(subset=FEATURE_COLS + ['target'])
        valid = valid.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLS)

        if len(valid) > 50:
            all_processed.append(valid)
            symbol_counts[symbol] = len(valid)
    except Exception as e:
        pass

combined = pd.concat(all_processed)
combined = combined.sort_index()

print(f"\nTotal samples: {len(combined):,}")
print(f"Stocks used: {len(all_processed)}")
print(f"Target distribution: {combined['target'].value_counts().to_dict()}")
print(f"Positive rate: {combined['target'].mean():.1%}")
print(f"Date range: {combined.index.min()} to {combined.index.max()}")

## 5. Walk-Forward Validation

Split chronologically — train on past, test on future. No data leakage.

In [ ]:
X = combined[FEATURE_COLS].values
y = combined['target'].values

# Chronological split: 70% train, 15% validation, 15% holdout
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_holdout, y_holdout = X[val_end:], y[val_end:]

print(f"Train: {len(X_train):,} samples ({combined.index[0]} to {combined.index[train_end-1]})")
print(f"Validation: {len(X_val):,} samples ({combined.index[train_end]} to {combined.index[val_end-1]})")
print(f"Holdout: {len(X_holdout):,} samples ({combined.index[val_end]} to {combined.index[-1]})")
print(f"\nTrain positive rate: {y_train.mean():.1%}")
print(f"Val positive rate: {y_val.mean():.1%}")
print(f"Holdout positive rate: {y_holdout.mean():.1%}")

## 6. CatBoost Training

In [ ]:
# Class balance check
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count
print(f"Train: {pos_count:,} positive / {neg_count:,} negative ({pos_count/len(y_train):.1%} positive)")
print(f"Val:   {y_val.sum():,} positive / {len(y_val)-y_val.sum():,} negative ({y_val.mean():.1%} positive)")

# Try GPU first, fall back to CPU
try:
    task_type = 'GPU'
    test_model = CatBoostClassifier(iterations=5, task_type='GPU', verbose=0)
    test_model.fit(X_train[:100], y_train[:100])
    print("Using GPU for training")
except Exception:
    task_type = 'CPU'
    print("GPU not available, using CPU")

params = dict(
    iterations=5000,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=5,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    min_data_in_leaf=30,
    loss_function='Logloss',
    eval_metric='Logloss',
    custom_metric=['F1', 'Precision', 'Recall'],
    task_type=task_type,
    random_seed=42,
    early_stopping_rounds=300,
    verbose=200,
)
if task_type == 'CPU':
    params['colsample_bylevel'] = 0.8

model = CatBoostClassifier(**params)

train_pool = Pool(X_train, y_train, feature_names=FEATURE_COLS)
val_pool = Pool(X_val, y_val, feature_names=FEATURE_COLS)

print(f"\nTraining CatBoost on {len(X_train):,} samples...")
model.fit(train_pool, eval_set=val_pool, use_best_model=True)

print(f"\nBest iteration: {model.get_best_iteration()}")
print(f"Best val Logloss: {model.get_best_score()['validation']['Logloss']:.4f}")

## 7. Evaluate on Holdout (Unseen Future Data)

In [ ]:
# Holdout evaluation
y_pred_holdout = model.predict(X_holdout)
y_prob_holdout = model.predict_proba(X_holdout)[:, 1]

print("=" * 60)
print("  HOLDOUT RESULTS (unseen future data)")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_holdout, y_pred_holdout):.1%}")
print(f"Precision: {precision_score(y_holdout, y_pred_holdout, zero_division=0):.1%}")
print(f"Recall:    {recall_score(y_holdout, y_pred_holdout, zero_division=0):.1%}")
print(f"F1 Score:  {f1_score(y_holdout, y_pred_holdout, zero_division=0):.1%}")
print()
print(classification_report(y_holdout, y_pred_holdout, target_names=['DOWN/FLAT', 'UP ≥1%']))

# Also check at different confidence thresholds
print("\n--- Performance at Different Confidence Thresholds ---")
for threshold in [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]:
    preds = (y_prob_holdout >= threshold).astype(int)
    n_signals = preds.sum()
    if n_signals > 0:
        prec = precision_score(y_holdout, preds, zero_division=0)
        rec = recall_score(y_holdout, preds, zero_division=0)
        f1 = f1_score(y_holdout, preds, zero_division=0)
        print(f"  Threshold {threshold:.0%}: {n_signals:>6,} signals | Precision {prec:.1%} | Recall {rec:.1%} | F1 {f1:.1%}")
    else:
        print(f"  Threshold {threshold:.0%}: 0 signals")

In [ ]:
# Validation set evaluation
y_pred_val = model.predict(X_val)
print("=" * 60)
print("  VALIDATION RESULTS")
print("=" * 60)
print(f"Accuracy:  {accuracy_score(y_val, y_pred_val):.1%}")
print(f"Precision: {precision_score(y_val, y_pred_val, zero_division=0):.1%}")
print(f"Recall:    {recall_score(y_val, y_pred_val, zero_division=0):.1%}")
print(f"F1 Score:  {f1_score(y_val, y_pred_val, zero_division=0):.1%}")

## 8. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance = model.get_feature_importance()
feature_imp = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': importance
}).sort_values('importance', ascending=False)

print("Top 20 Features:")
print(feature_imp.head(20).to_string(index=False))

plt.figure(figsize=(12, 10))
plt.barh(feature_imp['feature'].head(25)[::-1], feature_imp['importance'].head(25)[::-1])
plt.xlabel('Importance')
plt.title('CatBoost Feature Importance (Top 25)')
plt.tight_layout()
plt.show()

## 9. Retrain on Full Data (Train + Validation) for Deployment

In [ ]:
# Retrain on train + validation for deployment
X_deploy = np.concatenate([X_train, X_val])
y_deploy = np.concatenate([y_train, y_val])

best_iterations = max(model.get_best_iteration(), 300)

deploy_params = dict(
    iterations=best_iterations + 200,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=5,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    min_data_in_leaf=30,
    loss_function='Logloss',
    eval_metric='Logloss',
    task_type=task_type,
    random_seed=42,
    verbose=200,
)
if task_type == 'CPU':
    deploy_params['colsample_bylevel'] = 0.8

final_model = CatBoostClassifier(**deploy_params)

deploy_pool = Pool(X_deploy, y_deploy, feature_names=FEATURE_COLS)
holdout_pool = Pool(X_holdout, y_holdout, feature_names=FEATURE_COLS)

print(f"Retraining on {len(X_deploy):,} samples for deployment...")
final_model.fit(deploy_pool, eval_set=holdout_pool)

y_final_pred = final_model.predict(X_holdout)
print(f"\nFinal model holdout accuracy: {accuracy_score(y_holdout, y_final_pred):.1%}")
print(f"Final model holdout F1: {f1_score(y_holdout, y_final_pred, zero_division=0):.1%}")

## 10. Save Model for ai-trading-agent

In [ ]:
import hashlib

# Save CatBoost native format (smaller, faster loading)
final_model.save_model('predictor_catboost.cbm')

# Also save as pickle for compatibility with existing predictor.py
with open('predictor_catboost.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# Save feature columns list
with open('feature_cols.json', 'w') as f:
    json.dump({
        'features': FEATURE_COLS,
        'target_horizon': TARGET_HORIZON,
        'target_threshold': TARGET_THRESHOLD,
        'model_type': 'catboost',
        'training_samples': len(X_deploy),
        'stocks_count': len(all_processed),
        'interval': '1d',
        'holdout_f1': round(f1_score(y_holdout, y_final_pred, zero_division=0) * 100, 1),
        'holdout_precision': round(precision_score(y_holdout, y_final_pred, zero_division=0) * 100, 1),
        'holdout_accuracy': round(accuracy_score(y_holdout, y_final_pred) * 100, 1),
    }, f, indent=2)

# Compute SHA256 for integrity check
sha = hashlib.sha256()
with open('predictor_catboost.pkl', 'rb') as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha.update(chunk)
with open('predictor_catboost.pkl.sha256', 'w') as f:
    f.write(sha.hexdigest())

pkl_size = os.path.getsize('predictor_catboost.pkl') / (1024 * 1024)
cbm_size = os.path.getsize('predictor_catboost.cbm') / (1024 * 1024)

print(f"Saved: predictor_catboost.pkl ({pkl_size:.1f} MB)")
print(f"Saved: predictor_catboost.cbm ({cbm_size:.1f} MB)")
print(f"Saved: feature_cols.json")
print(f"Saved: predictor_catboost.pkl.sha256")
print(f"\nDownload these files and place in ai-trading-agent/models/")

## 11. Quick Sanity Check — Simulate Predictions

In [ ]:
# Simulate predictions on holdout — check if high-confidence signals are actually profitable
y_prob_final = final_model.predict_proba(X_holdout)[:, 1]

holdout_df = combined.iloc[val_end:].copy()
holdout_df['pred_prob'] = y_prob_final
holdout_df['pred_signal'] = (y_prob_final >= 0.65).astype(int)

# For high-confidence BUY signals, what's the actual outcome?
high_conf = holdout_df[holdout_df['pred_prob'] >= 0.65]
if len(high_conf) > 0:
    actual_wins = high_conf['target'].sum()
    total_signals = len(high_conf)
    win_rate = actual_wins / total_signals
    print(f"High-confidence signals (>=65%): {total_signals:,}")
    print(f"Actual win rate: {win_rate:.1%}")
    print(f"Expected: stock goes up >=1.5% within 3 trading days")

    # Breakdown by confidence bucket
    print("\n--- Win Rate by Confidence Bucket ---")
    for lo, hi in [(0.50, 0.55), (0.55, 0.60), (0.60, 0.65), (0.65, 0.70), (0.70, 0.75), (0.75, 0.80), (0.80, 1.0)]:
        bucket = holdout_df[(holdout_df['pred_prob'] >= lo) & (holdout_df['pred_prob'] < hi)]
        if len(bucket) > 10:
            wr = bucket['target'].mean()
            print(f"  {lo:.0%}-{hi:.0%}: {len(bucket):>6,} signals, win rate {wr:.1%}")
else:
    print("No high-confidence signals found in holdout.")